# Step 4 - Evaluate: test split, unseen Mirai run, per device type, explanations

1. Trains the two models still missing, with the same checkpointed training as step 3:
   * `xgb_strict`: portable without TCP window size / scale factor / options (fields that also fingerprint the sender's operating system);
   * `xgb_portable_mirai_holdout`: portable, trained **without the third Mirai run**.
2. Builds the test table once: every test packet of every file, copies stored in two files counted once.
3. Scores `xgb_portable` (main), `xgb_full`, `lgbm_portable`, `xgb_strict` and the 17-label model on the test split: per-class precision / recall / F1, benign packets flagged per million, attack packets missed, per device type.
4. Unseen-run test: the hold-out model on the third Mirai run, next to the model that saw that run's first 60%.
5. TreeSHAP explanations of the main model, including why some scanning packets are called brute force.

* **Runtime:** T4 GPU helps for the two trainings; scoring runs on CPU. About 45-60 min in total.
* **Disconnected?** Reconnect and run cells 1, 2 and 4 again: finished trainings, the test table and every finished score are kept.
* **Results:** copy what cell 6 prints into the chat.


In [ ]:
#@title 1. Settings
REPO_URL = "https://github.com/madhumithakk01/ds.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
ZIP_PATH = "/content/drive/MyDrive/Ds/GothamDataset2025.zip"  # "Ds" = shortcut to the shared folder  #@param {type:"string"}
WORK_DIR = "/content/drive/MyDrive/gotham_ids"  #@param {type:"string"}

In [ ]:
#@title 2. Mount Drive, get the latest code, check the environment
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/ds"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=True)

os.chdir(REPO_DIR)
if not os.path.exists(ZIP_PATH):
    folder = os.path.dirname(ZIP_PATH)
    print(f"ZIP_PATH not found: {ZIP_PATH}")
    if os.path.isdir(folder):
        print(f"Contents of {folder}:")
        for name in sorted(os.listdir(folder))[:50]:
            print("  ", name)
    print("Set ZIP_PATH in cell 1 to the zip shown above (the code will also search My Drive for it).")
os.environ["GIDS_WORK_DIR"] = WORK_DIR
os.environ["GIDS_ZIP_PATH"] = ZIP_PATH
print(subprocess.run(["git", "log", "-1", "--format=code version %h (%cd)"], capture_output=True, text=True).stdout)
!python -m gids env

In [ ]:
#@title 3. Self-test on a small synthetic archive (about 2 min)
!python -m gids selftest


In [ ]:
#@title 4. Evaluate (resumable: after a disconnect run cells 1, 2 and 4 again)
!python -u -m gids evaluate


In [ ]:
#@title 5. Report
from IPython.display import Markdown, display
display(Markdown(open(f"{WORK_DIR}/reports/evaluate/EVALUATE.md").read()))


In [ ]:
#@title 6. Plain text of the report (copy all of it into the chat)
print(open(f"{WORK_DIR}/reports/evaluate/EVALUATE.md").read())


### If something goes wrong
* `status` lists the trained models and which evaluation parts are finished.
* `trace --errors` prints warnings and errors of the latest run with full tracebacks.
* Results per model are in `MyDrive/gotham_ids/reports/evaluate/` (`test_<model>.json`, `unseen_run.json`, `explain.json`, `test_results.csv`).


In [ ]:
!python -m gids status
!python -m gids trace --errors --tail 30
